In [ ]:
!pip install bert-score

In [ ]:
# ============================================================
# Evaluate Similarity Between Human Proofs and GPT Proofs
# Metrics:
# 1. Lexical similarity: TF-IDF cosine similarity
# 2. Semantic similarity: Sentence-BERT cosine similarity
# 3. Contextual semantic overlap: BERTScore F1
# ============================================================

# Install required packages if needed:
# pip install pandas openpyxl scikit-learn sentence-transformers bert-score torch

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from bert_score import score


# ------------------------------------------------------------
# 1. Load the Excel file
# ------------------------------------------------------------

input_file = "/content/data-13.xlsx"   # Change path if needed
sheet_name = "Sheet1"

df = pd.read_excel(input_file, sheet_name=sheet_name)

# Clean column names
df.columns = df.columns.str.strip()

# Your file contains these columns:
human_col = "Human Proofs"
gpt_col = "GPT Proofs"

# Keep only rows where both Human and GPT proofs exist
df = df.dropna(subset=[human_col, gpt_col]).copy()

# Convert to strings
human_texts = df[human_col].astype(str).tolist()
gpt_texts = df[gpt_col].astype(str).tolist()


# ------------------------------------------------------------
# 2. Lexical similarity: TF-IDF cosine similarity
# ------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

all_texts = human_texts + gpt_texts
tfidf_matrix = tfidf_vectorizer.fit_transform(all_texts)

human_tfidf = tfidf_matrix[:len(human_texts)]
gpt_tfidf = tfidf_matrix[len(human_texts):]

lexical_scores = []

for i in range(len(human_texts)):
    score_i = cosine_similarity(human_tfidf[i], gpt_tfidf[i])[0][0]
    lexical_scores.append(score_i)

df["Lexical_TFIDF_Cosine"] = lexical_scores


# ------------------------------------------------------------
# 3. Semantic similarity: Sentence-BERT cosine similarity
# ------------------------------------------------------------

# You may also use:
# model_name = "sentence-transformers/all-mpnet-base-v2"
# model_name = "sentence-transformers/all-MiniLM-L6-v2"

model_name = "sentence-transformers/all-MiniLM-L6-v2"
sbert_model = SentenceTransformer(model_name)

human_embeddings = sbert_model.encode(
    human_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

gpt_embeddings = sbert_model.encode(
    gpt_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

semantic_scores = np.sum(human_embeddings * gpt_embeddings, axis=1)

df["Semantic_SBERT_Cosine"] = semantic_scores



def truncate_text(text, max_words=350):
    words = str(text).split()
    return " ".join(words[:max_words])

human_texts_bert = [truncate_text(t, max_words=350) for t in human_texts]
gpt_texts_bert = [truncate_text(t, max_words=350) for t in gpt_texts]

P, R, F1 = score(
    cands=gpt_texts_bert,
    refs=human_texts_bert,
    model_type="bert-base-uncased",
    lang="en",
    batch_size=4,
    verbose=True,
    use_fast_tokenizer=False
)

df["BERTScore_Precision"] = P.numpy()
df["BERTScore_Recall"] = R.numpy()
df["Contextual_BERTScore_F1"] = F1.numpy()

# ------------------------------------------------------------
# 4. Contextual semantic overlap: BERTScore F1
# Safe version for long proof texts
# ------------------------------------------------------------

from bert_score import score

# Clean texts again
human_texts = [str(x) for x in human_texts]
gpt_texts = [str(x) for x in gpt_texts]

P, R, F1 = score(
    cands=gpt_texts,
    refs=human_texts,
    model_type="bert-base-uncased",   # safer than SciBERT for this error
    lang="en",
    batch_size=4,
    verbose=True,
    use_fast_tokenizer=False          # helps avoid tokenizer overflow errors
)

df["BERTScore_Precision"] = P.numpy()
df["BERTScore_Recall"] = R.numpy()
df["Contextual_BERTScore_F1"] = F1.numpy()

# ------------------------------------------------------------
# 5. Create summary table
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Metric": [
        "Lexical similarity: TF-IDF cosine",
        "Semantic similarity: Sentence-BERT cosine",
        "Contextual semantic overlap: BERTScore F1"
    ],
    "Mean": [
        df["Lexical_TFIDF_Cosine"].mean(),
        df["Semantic_SBERT_Cosine"].mean(),
        df["Contextual_BERTScore_F1"].mean()
    ],
    "Median": [
        df["Lexical_TFIDF_Cosine"].median(),
        df["Semantic_SBERT_Cosine"].median(),
        df["Contextual_BERTScore_F1"].median()
    ],
    "Std": [
        df["Lexical_TFIDF_Cosine"].std(),
        df["Semantic_SBERT_Cosine"].std(),
        df["Contextual_BERTScore_F1"].std()
    ],
    "Min": [
        df["Lexical_TFIDF_Cosine"].min(),
        df["Semantic_SBERT_Cosine"].min(),
        df["Contextual_BERTScore_F1"].min()
    ],
    "Max": [
        df["Lexical_TFIDF_Cosine"].max(),
        df["Semantic_SBERT_Cosine"].max(),
        df["Contextual_BERTScore_F1"].max()
    ]
})


# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

output_file = "proof_similarity_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Row_Level_Metrics", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)

print("Analysis completed successfully.")
print(f"Results saved to: {output_file}")

print("\nSummary:")
print(summary)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/498 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/498 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/250 [00:00<?, ?it/s]

done in 1412.79 seconds, 0.71 sentences/sec
Analysis completed successfully.
Results saved to: proof_similarity_results.xlsx

Summary:
                                      Metric      Mean    Median       Std  \
0          Lexical similarity: TF-IDF cosine  0.270861  0.228874  0.201505   
1  Semantic similarity: Sentence-BERT cosine  0.717758  0.733671  0.133410   
2  Contextual semantic overlap: BERTScore F1  0.755012  0.756405  0.054043   

        Min       Max  
0  0.001313  0.940050  
1 -0.197027  0.968346  
2  0.571696  0.901496  
